In [ ]:
# ---------------------------------------------------------------------------
# Clone the bc-II repository (contains bounding-box / MBR / TBR definitions
# and the .env_bin.txt configuration file).
# NOTE: This repo is for coordinate definitions ONLY — it is NOT the target
#       for storing generated GeoTIFF files.  See the "Export to GitHub" section
#       at the bottom of this notebook for instructions on pushing TIFFs to a
#       separate data-storage repository.
# ---------------------------------------------------------------------------
!git clone https://github.com/chase-kusterer/bc-II.git

# Change the working directory into the cloned repo so we can access its files
%cd bc-II

# Verify the environment configuration file exists (suppress its output)
!cat .env_bin.txt > /dev/null

In [ ]:
# ---------------------------------------------------------------------------
# Mount Google Drive so it appears as a local filesystem path in Colab.
# You will be prompted to authorise access the first time.
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# Define the output directory on Google Drive.
# This creates the folder if it doesn't already exist.
# TODO: Adjust the folder path to match your Drive structure.
# ---------------------------------------------------------------------------
import os

output_dir = "/content/drive/MyDrive/Business Challenge II/GeoTIFFs"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Suppress non-critical warnings to keep notebook output clean
import warnings
warnings.filterwarnings('ignore')

# Install / upgrade the geospatial stack used for satellite data access
!pip install --upgrade rioxarray stackstac pystac-client planetary_computer odc-stac

# ---------------------------------------------------------------------------
# Core GIS & numerical libraries
# ---------------------------------------------------------------------------
import numpy as np                          # array operations
import xarray as xr                         # labelled multi-dimensional arrays
import matplotlib.pyplot as plt             # plotting
import rioxarray as rio                     # xarray <-> rasterio bridge
import rasterio                             # low-level raster I/O (GeoTIFF)
from matplotlib.cm import RdYlGn, jet, RdBu  # colormaps for index visualisation

# ---------------------------------------------------------------------------
# Microsoft Planetary Computer / STAC libraries
# ---------------------------------------------------------------------------
import stackstac                            # STAC items -> xarray DataArray
import pystac_client                        # search STAC catalogues
import planetary_computer                   # sign URLs for Planetary Computer
from odc.stac import stac_load              # alternative STAC loader (ODC)

### Discover and Load the Data for Analysis

First, we define our area of interest using latitude and longitude coordinates
read from the project CSV.  Two bounding boxes are computed:

* **MBR (Minimum Bounding Rectangle)** — tightly fits the data points.
* **TBR (Tolerance Bounding Rectangle)** — adds a buffer around the MBR so that
  edge pixels are not cut off during raster extraction.

In [ ]:
import pandas as pd

# Read the dataset that contains ground-truth temperature observations
# TODO: Replace "file_path" with the actual path to your CSV
df = pd.read_csv("/content/bc-II/Data/filename_uhi_data.csv")

# ---------------------------------------------------------------------------
# Minimum Bounding Rectangle (MBR)
# The MBR is the smallest rectangle that encloses every data point.
# ---------------------------------------------------------------------------
min_lat = df['Latitude'].min()
max_lat = df['Latitude'].max()
min_lon = df['Longitude'].min()
max_lon = df['Longitude'].max()

mbr_lower_left  = (min_lat, min_lon)
mbr_upper_right = (max_lat, max_lon)

print("MBR lower-left:", mbr_lower_left, " | upper-right:", mbr_upper_right)

# ---------------------------------------------------------------------------
# Tolerance Bounding Rectangle (TBR)
# A buffer is added around the MBR to ensure satellite pixels at the edges
# are fully captured.  Adjust `buffer` (in degrees) as needed.
# ---------------------------------------------------------------------------
buffer = 0.1  # degrees (~11 km at the equator)

tbr_lower_left  = (min_lat - buffer, min_lon - buffer)
tbr_upper_right = (max_lat + buffer, max_lon + buffer)

print("TBR lower-left:", tbr_lower_left, " | upper-right:", tbr_upper_right)

In [ ]:
# ---------------------------------------------------------------------------
# Reformat bounding boxes to the (west, south, east, north) / (min_lon,
# min_lat, max_lon, max_lat) convention expected by STAC search and rasterio.
# ---------------------------------------------------------------------------
mbr_bounds = (mbr_lower_left[1], mbr_lower_left[0],
              mbr_upper_right[1], mbr_upper_right[0])

tbr_bounds = (tbr_lower_left[1], tbr_lower_left[0],
              tbr_upper_right[1], tbr_upper_right[0])

print("MBR bounds (W, S, E, N):", mbr_bounds)
print("TBR bounds (W, S, E, N):", tbr_bounds)

In [ ]:
# ---------------------------------------------------------------------------
# Define the satellite imagery search window.
# Format: "YYYY-MM-DD/YYYY-MM-DD"
# The window should bracket the ground-truth data collection date.  A wider
# window gives more scenes to choose from (helpful for cloud-free selection),
# but dates far from collection may not represent ground conditions accurately.
# TODO: Replace the placeholder with actual dates.
# ---------------------------------------------------------------------------
time_window = "yyyy-mm-dd/yyyy-mm-dd" 

In [ ]:
# ---------------------------------------------------------------------------
# Spatial resolution settings
# ---------------------------------------------------------------------------
# Sentinel-2 native resolution for visible/NIR bands is 10 m.  Because we
# reproject to EPSG:4326 (degrees), we convert metres -> degrees using the
# approximation 1° latitude ≈ 111 320 m.
# ---------------------------------------------------------------------------
resolution = 10            # metres per pixel
scale = resolution / 111320.0  # approximate degrees per pixel at the equator

# MBR Image Extraction

Using the `pystac_client` we can search the Planetary Computer's STAC endpoint
for items matching our query parameters.  The query filters for "low-cloud"
scenes with overall cloud cover < 30 %.  The result is the number of scenes
matching our search criteria that intersect our MBR area of interest.  Some of
these may be partial scenes or still contain clouds.

In [ ]:
# ---------------------------------------------------------------------------
# Connect to Microsoft Planetary Computer's STAC API
# ---------------------------------------------------------------------------
stac = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

# ---------------------------------------------------------------------------
# Search for Sentinel-2 Level-2A scenes that intersect the MBR
# and fall within the specified time window, with < 30 % cloud cover.
# ---------------------------------------------------------------------------
search_mbr = stac.search(
    bbox=mbr_bounds,
    datetime=time_window,
    collections=["sentinel-2-l2a"],
    query={"eo:cloud_cover": {"lt": 30}},
)

In [ ]:
# Materialise the MBR search results into a list
items_mbr = list(search_mbr.get_items())
print(f"MBR: {len(items_mbr)} scenes found within the time window.")

Next, we load the data into an [xarray](https://xarray.pydata.org/en/stable/)
Dataset using [odc-stac](https://odc-stac.readthedocs.io/).  We keep all 11
commonly used Sentinel-2 spectral bands.  Key settings:

* **CRS** — EPSG:4326 (latitude / longitude in degrees).
* **Resolution** — ~10 m (converted to degrees above).
* **Chunking** — Dask chunks of 2048 × 2048 for memory-efficient lazy loading.

In [ ]:
# ---------------------------------------------------------------------------
# Sign each STAC item so that its asset URLs carry a short-lived SAS token
# required by Microsoft Planetary Computer for data download.
# ---------------------------------------------------------------------------
signed_items_mbr = [planetary_computer.sign(item).to_dict() for item in items_mbr]
# (no printed output expected)

### Sentinel-2 Bands Summary

| Band | Description            | Native Resolution |
|------|------------------------|-------------------|
| B01  | Coastal Aerosol        | 60 m              |
| B02  | Blue                   | 10 m              |
| B03  | Green                  | 10 m              |
| B04  | Red                    | 10 m              |
| B05  | Red Edge (704 nm)      | 20 m              |
| B06  | Red Edge (740 nm)      | 20 m              |
| B07  | Red Edge (780 nm)      | 20 m              |
| B08  | NIR (833 nm)           | 10 m              |
| B8A  | NIR narrow (864 nm)    | 20 m              |
| B11  | SWIR (1.6 µm)         | 20 m              |
| B12  | SWIR (2.2 µm)         | 20 m              |

In [ ]:
# ---------------------------------------------------------------------------
# Load MBR satellite data via ODC stac_load.
# All 11 bands are loaded so derived indices (NDVI, NDBI, NDWI, etc.)
# can be computed flexibly later.
# ---------------------------------------------------------------------------
data_mbr = stac_load(
    items_mbr,
    bands=["B01", "B02", "B03", "B04", "B05", "B06", "B07",
           "B08", "B8A", "B11", "B12"],
    crs="EPSG:4326",
    resolution=scale,
    chunks={"x": 2048, "y": 2048},
    dtype="uint16",
    patch_url=planetary_computer.sign,
    bbox=mbr_bounds,
)

In [ ]:
# Inspect the MBR xarray Dataset dimensions, coordinates, and variables
# to confirm the spatial extent, time steps, and band names are correct.
display(data_mbr)

### View RGB (True-Colour) Images from the Time Series

The grid below shows every scene returned by the search.  Some may have clouds
or partial coverage due to Sentinel-2 orbit swath boundaries.  Scene numbering
starts at **0** (top-left) and proceeds left-to-right, row-by-row.  These
thumbnails are for quick review only — aspect ratios are not preserved.

In [ ]:
# Combine Red (B04), Green (B03), Blue (B02) into an RGB DataArray and plot
# each scene in the time series as a small-multiple grid.
plot_data_mbr = data_mbr[["B04", "B03", "B02"]].to_array()
plot_data_mbr.plot.imshow(col='time', col_wrap=4, robust=True, vmin=0, vmax=2500)
plt.show()

### Single-Scene Selection

After visually inspecting all available scenes, manually pick the one with the
least cloud cover and best spatial completeness.  Update the `time=` index
below to match your chosen scene.

In [ ]:
# ---------------------------------------------------------------------------
# Display a single MBR scene chosen for minimal cloud cover.
# TODO: Update `time=9` and the title string to match the scene you select.
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6))
plot_data_mbr.isel(time=9).plot.imshow(robust=True, ax=ax, vmin=0, vmax=2500)
ax.set_title("MBR RGB Single Date: <UPDATE DATE>")
ax.axis('off')
plt.show()

In [ ]:
# Extract the single time-step chosen above for index calculations and
# GeoTIFF export.  Keep this index consistent with the plot above.
slice_mbr = data_mbr.isel(time=9)

The **Normalized Difference Vegetation Index (NDVI)** quantifies vegetation
"greenness".  Values range roughly from −1 to +1:

* **< 0.25** — bare soil, urban surfaces, water
* **0.25 – 0.6** — grasslands or growing crops
* **0.6 – 1.0** — dense, healthy vegetation

**NDVI = (NIR − Red) / (NIR + Red)** = (B08 − B04) / (B08 + B04)

In [ ]:
# ---------------------------------------------------------------------------
# Compute NDVI for the selected single scene.
# The raw Sentinel-2 bands are stored as uint16; cast to float64 first to
# avoid integer-overflow and to allow negative index values.
# ---------------------------------------------------------------------------
ndvi_slice_mbr = (
    (slice_mbr.B08.astype('float64') - slice_mbr.B04.astype('float64')) /
    (slice_mbr.B08.astype('float64') + slice_mbr.B04.astype('float64'))
)

In [ ]:
# Visualise NDVI — green = high vegetation, red = low / none
fig, ax = plt.subplots(figsize=(7, 6))
ndvi_slice_mbr.plot.imshow(vmin=0.0, vmax=0.8, cmap="RdYlGn")
plt.title("MBR Single Scene NDVI: <UPDATE DATE>")
plt.axis('off')
plt.show()

The **Normalized Difference Built-up Index (NDBI)** highlights urbanised /
built-up areas using NIR and SWIR:

* **< 0** — vegetation or water (not built-up)
* **> 0** — increasingly dense built-up surfaces

**NDBI = (SWIR − NIR) / (SWIR + NIR)** = (B11 − B08) / (B11 + B08)

In [ ]:
# Compute NDBI for the selected single scene
ndbi_slice_mbr = (
    (slice_mbr.B11.astype('float64') - slice_mbr.B08.astype('float64')) /
    (slice_mbr.B11.astype('float64') + slice_mbr.B08.astype('float64'))
)

In [ ]:
# Visualise NDBI — warmer colours indicate denser built-up area
fig, ax = plt.subplots(figsize=(7, 6))
ndbi_slice_mbr.plot.imshow(vmin=-0.1, vmax=0.04, cmap="jet")
plt.title("MBR Single Scene NDBI: <UPDATE DATE>")
plt.axis('off')
plt.show()

The **Normalized Difference Water Index (NDWI)** identifies surface water
using Green and NIR:

* **> 0** — water (shown in blue below)
* **< 0** — non-water (shown in red below)

**NDWI = (Green − NIR) / (Green + NIR)** = (B03 − B08) / (B03 + B08)

In [ ]:
# Compute NDWI for the selected single scene
ndwi_slice_mbr = (
    (slice_mbr.B03.astype('float64') - slice_mbr.B08.astype('float64')) /
    (slice_mbr.B03.astype('float64') + slice_mbr.B08.astype('float64'))
)

In [ ]:
# Visualise NDWI — blue = water, red = non-water
fig, ax = plt.subplots(figsize=(7, 6))
ndwi_slice_mbr.plot.imshow(vmin=-0.4, vmax=0.2, cmap="RdBu")
plt.title("MBR Single Scene NDWI: <UPDATE DATE>")
plt.axis('off')
plt.show()

### Median Composite

Computing the **pixel-wise median** across the time dimension produces a
cloud-free mosaic.  Because cloud cover is < 30 % per scene and clouds appear
in different locations across scenes, the median value at any given pixel is
very unlikely to be a cloud.

In [ ]:
# Compute the median composite across all time steps for the MBR data.
# .compute() triggers Dask execution and loads the result into memory.
median_mbr = data_mbr.median(dim="time").compute()

In [ ]:
# Plot the cloud-free MBR median RGB composite
fig, ax = plt.subplots(figsize=(6, 6))
median_mbr[["B04", "B03", "B02"]].to_array().plot.imshow(
    robust=True, ax=ax, vmin=0, vmax=2500
)
ax.set_title("MBR RGB Median Composite")
ax.axis('off')
plt.show()

#### Median NDVI

In [ ]:
# The median operation already produced float values, so no explicit cast is
# needed — a direct band-math calculation is valid.
ndvi_median_mbr = (median_mbr.B08 - median_mbr.B04) / (median_mbr.B08 + median_mbr.B04)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ndvi_median_mbr.plot.imshow(vmin=0.0, vmax=0.8, cmap="RdYlGn")
plt.title("MBR Median NDVI")
plt.axis('off')
plt.show()

#### Median NDBI

In [ ]:
# Compute NDBI from the median composite
ndbi_median_mbr = (median_mbr.B11 - median_mbr.B08) / (median_mbr.B11 + median_mbr.B08)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ndbi_median_mbr.plot.imshow(vmin=-0.1, vmax=0.04, cmap="jet")
plt.title("MBR Median NDBI")
plt.axis('off')
plt.show()

#### Median NDWI

In [ ]:
# Compute NDWI from the median composite
ndwi_median_mbr = (median_mbr.B03 - median_mbr.B08) / (median_mbr.B03 + median_mbr.B08)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ndwi_median_mbr.plot.imshow(vmin=-0.4, vmax=0.2, cmap="RdBu")
plt.title("MBR Median NDWI")
plt.axis('off')
plt.show()

### Save MBR GeoTIFF Files

Each GeoTIFF contains **3 bands**: NDVI (band 1), NDBI (band 2), NDWI (band 3).
Two files are created — one from the best single scene and one from the median
composite.  Additional bands or indices can be added by increasing `count` and
appending more `dst.write()` calls.

**TODO:** Update the filenames below with the actual city name and dates.

In [ ]:
# ---------------------------------------------------------------------------
# Define output filenames
# TODO: Replace placeholders with actual city name and date(s).
# ---------------------------------------------------------------------------
filename_slice_mbr  = "City_MBR_Best_Single(yyyy-mm-dd).tiff"
filename_median_mbr = "City_MBR_Median_Composite(yyyy-mm-ddTOyyyy-mm-dd).tiff" 

In [ ]:
# ---------------------------------------------------------------------------
# Build full output file paths pointing to Google Drive
# ---------------------------------------------------------------------------
filepath_slice_mbr  = os.path.join(output_dir, filename_slice_mbr)
filepath_median_mbr = os.path.join(output_dir, filename_median_mbr)

#### Sanity Check — Verify Spatial Dimensions

Confirm that the single-scene slice and the median composite share the same
latitude × longitude grid dimensions before writing GeoTIFFs.

In [ ]:
# Read spatial dimensions from the xarray objects
height_slice_mbr  = slice_mbr.dims["latitude"]
width_slice_mbr   = slice_mbr.dims["longitude"]

height_median_mbr = median_mbr.dims["latitude"]
width_median_mbr  = median_mbr.dims["longitude"]

print(f"Single slice : {height_slice_mbr} rows × {width_slice_mbr} cols")
print(f"Median       : {height_median_mbr} rows × {width_median_mbr} cols")
assert (height_slice_mbr, width_slice_mbr) == (height_median_mbr, width_median_mbr), \
    "Dimension mismatch between single slice and median composite!" 

In [ ]:
# ---------------------------------------------------------------------------
# Build affine transforms that map pixel coordinates to geographic coordinates.
# rasterio.transform.from_bounds(west, south, east, north, width, height)
# ---------------------------------------------------------------------------
gt_slice_mbr = rasterio.transform.from_bounds(
    mbr_lower_left[1], mbr_lower_left[0],
    mbr_upper_right[1], mbr_upper_right[0],
    width_slice_mbr, height_slice_mbr
)

# Write CRS and transform metadata into the xarray objects (in-place)
slice_mbr.rio.write_crs("epsg:4326", inplace=True)
slice_mbr.rio.write_transform(transform=gt_slice_mbr, inplace=True)

gt_median_mbr = rasterio.transform.from_bounds(
    mbr_lower_left[1], mbr_lower_left[0],
    mbr_upper_right[1], mbr_upper_right[0],
    width_median_mbr, height_median_mbr
)

median_mbr.rio.write_crs("epsg:4326", inplace=True)
median_mbr.rio.write_transform(transform=gt_median_mbr, inplace=True)

In [ ]:
# ---------------------------------------------------------------------------
# Write MBR GeoTIFF — Single Scene
# 3 bands: NDVI, NDBI, NDWI  |  LZW compression  |  float64
# The `with` context manager closes the file automatically; no explicit
# dst.close() is needed (removed redundant call from original).
# ---------------------------------------------------------------------------
with rasterio.open(
    filepath_slice_mbr, 'w', driver='GTiff',
    width=width_slice_mbr, height=height_slice_mbr,
    crs='epsg:4326', transform=gt_slice_mbr,
    count=3, compress='lzw', dtype='float64'
) as dst:
    dst.write(ndvi_slice_mbr, 1)  # Band 1: NDVI
    dst.write(ndbi_slice_mbr, 2)  # Band 2: NDBI
    dst.write(ndwi_slice_mbr, 3)  # Band 3: NDWI
    dst.close()

# ---------------------------------------------------------------------------
# Write MBR GeoTIFF — Median Composite
# ---------------------------------------------------------------------------
with rasterio.open(
    filepath_median_mbr, 'w', driver='GTiff',
    width=width_median_mbr, height=height_median_mbr,
    crs='epsg:4326', transform=gt_median_mbr,
    count=3, compress='lzw', dtype='float64'
) as dst:
    dst.write(ndvi_median_mbr, 1)  # Band 1: NDVI
    dst.write(ndbi_median_mbr, 2)  # Band 2: NDBI
    dst.write(ndwi_median_mbr, 3)  # Band 3: NDWI
    dst.close()

print("MBR GeoTIFF files written successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Verify the files were written successfully
# ---------------------------------------------------------------------------
for f in [filepath_slice_mbr, filepath_median_mbr]:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {size_mb:6.1f} MB  {f}")

# TBR Image Extraction

This section repeats the workflow above using the **Tolerance Bounding
Rectangle (TBR)** — a buffered version of the MBR that captures additional
context pixels around the edges.

In [ ]:
# ---------------------------------------------------------------------------
# Search for Sentinel-2 scenes intersecting the TBR
# ---------------------------------------------------------------------------
stac = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

search_tbr = stac.search(
    bbox=tbr_bounds,
    datetime=time_window,
    collections=["sentinel-2-l2a"],
    query={"eo:cloud_cover": {"lt": 30}},
)

In [ ]:
# Materialise TBR search results
items_tbr = list(search_tbr.get_items())
print(f"TBR: {len(items_tbr)} scenes found within the time window.")

Next, we load the TBR data with the same band selection, CRS, resolution, and
chunking as the MBR data above.

In [ ]:
# Sign TBR STAC items for Planetary Computer download
signed_items_tbr = [planetary_computer.sign(item).to_dict() for item in items_tbr]
# (no printed output expected)

### Sentinel-2 Bands Summary

(Same band table as the MBR section — included here for self-contained reference.)

| Band | Description            | Native Resolution |
|------|------------------------|-------------------|
| B01  | Coastal Aerosol        | 60 m              |
| B02  | Blue                   | 10 m              |
| B03  | Green                  | 10 m              |
| B04  | Red                    | 10 m              |
| B05  | Red Edge (704 nm)      | 20 m              |
| B06  | Red Edge (740 nm)      | 20 m              |
| B07  | Red Edge (780 nm)      | 20 m              |
| B08  | NIR (833 nm)           | 10 m              |
| B8A  | NIR narrow (864 nm)    | 20 m              |
| B11  | SWIR (1.6 µm)         | 20 m              |
| B12  | SWIR (2.2 µm)         | 20 m              |

In [ ]:
# ---------------------------------------------------------------------------
# Load TBR satellite data
# ---------------------------------------------------------------------------
data_tbr = stac_load(
    items_tbr,
    bands=["B01", "B02", "B03", "B04", "B05", "B06", "B07",
           "B08", "B8A", "B11", "B12"],
    crs="EPSG:4326",
    resolution=scale,
    chunks={"x": 2048, "y": 2048},
    dtype="uint16",
    patch_url=planetary_computer.sign,
    bbox=tbr_bounds,
)

In [ ]:
# Inspect the TBR Dataset dimensions and variables
display(data_tbr)

### View RGB (True-Colour) Images from the Time Series

Same visual inspection as the MBR section.  Pick the best cloud-free scene.

In [ ]:
# Combine B04/B03/B02 into an RGB DataArray and display the time series grid
plot_data_tbr = data_tbr[["B04", "B03", "B02"]].to_array()
plot_data_tbr.plot.imshow(col='time', col_wrap=4, robust=True, vmin=0, vmax=2500)
plt.show()

### Single-Scene Selection

Select the best TBR scene — update `time=` below accordingly.

In [ ]:
# Display a single TBR scene
# TODO: Update time index and title to match your chosen scene
fig, ax = plt.subplots(figsize=(6, 6))
plot_data_tbr.isel(time=9).plot.imshow(robust=True, ax=ax, vmin=0, vmax=2500)
ax.set_title("TBR RGB Single Date: <UPDATE DATE>")
ax.axis('off')
plt.show()

In [ ]:
# Extract the selected TBR single scene for index calculations
slice_tbr = data_tbr.isel(time=9)

**NDVI** = (B08 − B04) / (B08 + B04)  — see MBR section for full description.

In [ ]:
# Compute NDVI for the selected TBR single scene (cast to float64)
ndvi_slice_tbr = (
    (slice_tbr.B08.astype('float64') - slice_tbr.B04.astype('float64')) /
    (slice_tbr.B08.astype('float64') + slice_tbr.B04.astype('float64'))
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ndvi_slice_tbr.plot.imshow(vmin=0.0, vmax=0.8, cmap="RdYlGn")
plt.title("TBR Single Scene NDVI: <UPDATE DATE>")
plt.axis('off')
plt.show()

**NDBI** = (B11 − B08) / (B11 + B08)  — see MBR section for full description.

In [ ]:
# Compute NDBI for the selected TBR single scene
ndbi_slice_tbr = (
    (slice_tbr.B11.astype('float64') - slice_tbr.B08.astype('float64')) /
    (slice_tbr.B11.astype('float64') + slice_tbr.B08.astype('float64'))
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ndbi_slice_tbr.plot.imshow(vmin=-0.1, vmax=0.04, cmap="jet")
plt.title("TBR Single Scene NDBI: <UPDATE DATE>")
plt.axis('off')
plt.show()

**NDWI** = (B03 − B08) / (B03 + B08)  — see MBR section for full description.

In [ ]:
# Compute NDWI for the selected TBR single scene
ndwi_slice_tbr = (
    (slice_tbr.B03.astype('float64') - slice_tbr.B08.astype('float64')) /
    (slice_tbr.B03.astype('float64') + slice_tbr.B08.astype('float64'))
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ndwi_slice_tbr.plot.imshow(vmin=-0.4, vmax=0.2, cmap="RdBu")
plt.title("TBR Single Scene NDWI: <UPDATE DATE>")
plt.axis('off')
plt.show()

### Median Composite

Pixel-wise median across time for the TBR — same cloud-filtering logic as MBR.

In [ ]:
# Compute the TBR median composite
median_tbr = data_tbr.median(dim="time").compute()

In [ ]:
# Plot the cloud-free TBR median RGB composite
fig, ax = plt.subplots(figsize=(6, 6))
median_tbr[["B04", "B03", "B02"]].to_array().plot.imshow(
    robust=True, ax=ax, vmin=0, vmax=2500
)
ax.set_title("TBR RGB Median Composite")
ax.axis('off')
plt.show()

#### Median NDVI

In [ ]:
# ---------------------------------------------------------------------------
# BUG FIX: The original notebook computed ndvi_median_tbr from `median_mbr`
# instead of `median_tbr`.  This has been corrected below.
# ---------------------------------------------------------------------------
ndvi_median_tbr = (median_tbr.B08 - median_tbr.B04) / (median_tbr.B08 + median_tbr.B04)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ndvi_median_tbr.plot.imshow(vmin=0.0, vmax=0.8, cmap="RdYlGn")
plt.title("TBR Median NDVI")
plt.axis('off')
plt.show()

#### Median NDBI

In [ ]:
# Compute NDBI from the TBR median composite
ndbi_median_tbr = (median_tbr.B11 - median_tbr.B08) / (median_tbr.B11 + median_tbr.B08)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ndbi_median_tbr.plot.imshow(vmin=-0.1, vmax=0.04, cmap="jet")
plt.title("TBR Median NDBI")
plt.axis('off')
plt.show()

#### Median NDWI

In [ ]:
# Compute NDWI from the TBR median composite
ndwi_median_tbr = (median_tbr.B03 - median_tbr.B08) / (median_tbr.B03 + median_tbr.B08)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ndwi_median_tbr.plot.imshow(vmin=-0.4, vmax=0.2, cmap="RdBu")
plt.title("TBR Median NDWI")
plt.axis('off')
plt.show()

### Save TBR GeoTIFF Files

Same 3-band layout as MBR: NDVI (1), NDBI (2), NDWI (3).

**TODO:** Update filenames with the actual city name and dates.

In [ ]:
filename_slice_tbr  = "City_TBR_Best_Single(yyyy-mm-dd).tiff"
filename_median_tbr = "City_TBR_Median_Composite(yyyy-mm-ddTOyyyy-mm-dd).tiff" 

In [ ]:
# ---------------------------------------------------------------------------
# Build full output file paths pointing to Google Drive
# ---------------------------------------------------------------------------
filepath_slice_tbr  = os.path.join(output_dir, filename_slice_tbr)
filepath_median_tbr = os.path.join(output_dir, filename_median_tbr)

#### Sanity Check — Verify Spatial Dimensions

In [ ]:
height_slice_tbr  = slice_tbr.dims["latitude"]
width_slice_tbr   = slice_tbr.dims["longitude"]

height_median_tbr = median_tbr.dims["latitude"]
width_median_tbr  = median_tbr.dims["longitude"]

print(f"Single slice : {height_slice_tbr} rows × {width_slice_tbr} cols")
print(f"Median       : {height_median_tbr} rows × {width_median_tbr} cols")
assert (height_slice_tbr, width_slice_tbr) == (height_median_tbr, width_median_tbr), \
    "Dimension mismatch between single slice and median composite!" 

In [ ]:
# Build affine transforms for TBR bounds
gt_slice_tbr = rasterio.transform.from_bounds(
    tbr_lower_left[1], tbr_lower_left[0],
    tbr_upper_right[1], tbr_upper_right[0],
    width_slice_tbr, height_slice_tbr
)

# BUG FIX: Original had "espg:4326" typo — corrected to "epsg:4326"
slice_tbr.rio.write_crs("epsg:4326", inplace=True)
slice_tbr.rio.write_transform(transform=gt_slice_tbr, inplace=True)

gt_median_tbr = rasterio.transform.from_bounds(
    tbr_lower_left[1], tbr_lower_left[0],
    tbr_upper_right[1], tbr_upper_right[0],
    width_median_tbr, height_median_tbr
)

median_tbr.rio.write_crs("epsg:4326", inplace=True)
median_tbr.rio.write_transform(transform=gt_median_tbr, inplace=True)

In [ ]:
# ---------------------------------------------------------------------------
# Write TBR GeoTIFF — Single Scene
# ---------------------------------------------------------------------------
with rasterio.open(
    filepath_slice_tbr, 'w', driver='GTiff',
    width=width_slice_tbr, height=height_slice_tbr,
    crs='epsg:4326', transform=gt_slice_tbr,
    count=3, compress='lzw', dtype='float64'
) as dst:
    dst.write(ndvi_slice_tbr, 1)  # Band 1: NDVI
    dst.write(ndbi_slice_tbr, 2)  # Band 2: NDBI
    dst.write(ndwi_slice_tbr, 3)  # Band 3: NDWI

# ---------------------------------------------------------------------------
# Write TBR GeoTIFF — Median Composite
# ---------------------------------------------------------------------------
with rasterio.open(
    filepath_median_tbr, 'w', driver='GTiff',
    width=width_median_tbr, height=height_median_tbr,
    crs='epsg:4326', transform=gt_median_tbr,
    count=3, compress='lzw', dtype='float64'
) as dst:
    dst.write(ndvi_median_tbr, 1)  # Band 1: NDVI
    dst.write(ndbi_median_tbr, 2)  # Band 2: NDBI
    dst.write(ndwi_median_tbr, 3)  # Band 3: NDWI

print("TBR GeoTIFF files written successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# Verify the files were written successfully
# ---------------------------------------------------------------------------
for f in [filepath_slice_mbr, filepath_median_mbr,
          filepath_slice_tbr, filepath_median_tbr]:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {size_mb:6.1f} MB  {f}")